# PubMed pathogen category assignment

This notebook reads the accepted abstracts from notebook 02, classifies
abstract-level animal-infection and zoonosis evidence, and derives the
corpus-relative categories 1, 2, and 3. It writes a pathogen-level category
table under outputs/pubmed_screening/.

In [1]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'assets').exists():
    raise FileNotFoundError('Could not locate the repository assets directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graphicalizer import (
    OpenAIChatCompleter,
    derive_category_counts,
    load_corpus_articles,
    screen_pubmed_corpus,
)

## Load configuration for this notebook

In [2]:
from graphicalizer.notebook_config import (
    configured_env,
    debug_pathogens,
    limit_debug_abstracts,
    load_notebook_config,
    resolve_config_path,
)

CONFIG = load_notebook_config(PROJECT_ROOT)
COMMON = CONFIG['common']
SETTINGS = CONFIG['notebook_03_pubmed_pathogen_category']
DEBUG_MODE = COMMON['debug_mode']
DEBUG_PATHOGENS = debug_pathogens(COMMON)

OUTPUT_DIR = resolve_config_path(
    PROJECT_ROOT,
    Path(COMMON['output_root'])
    / COMMON['corpus_subdir']
    / ('debug' if DEBUG_MODE else ''),
)
REFINED_CORPUS_PATH = OUTPUT_DIR / SETTINGS['refined_corpus_filename']
CORPUS_PATHOGENS = (
    list(DEBUG_PATHOGENS)
    if DEBUG_MODE
    else COMMON['corpus_pathogens']
)
CORPUS_START_YEAR = COMMON['corpus_start_year']
CORPUS_END_YEAR = COMMON['corpus_end_year']
PATHOGENS = DEBUG_PATHOGENS if DEBUG_MODE else COMMON['pathogens']
OPENAI_MODEL = configured_env(COMMON, 'openai_model_env', COMMON['openai_model'])
OPENAI_API_KEY = configured_env(COMMON, 'openai_api_key_env')
LLM_MAX_TOKENS = SETTINGS['openai_max_tokens']
LLM_RETRIES = COMMON['llm_retries']
LLM_MAX_CALLS = COMMON['llm_max_calls']
LLM_SAVE_EVERY = COMMON['llm_save_every']
RESUME = COMMON['resume']
RETRY_FAILED_LLM_ROWS = COMMON['retry_failed_llm_rows']


## Load and filter the accepted corpus

In [3]:
corpus = load_corpus_articles(
    REFINED_CORPUS_PATH,
    pathogens=CORPUS_PATHOGENS,
    start_year=CORPUS_START_YEAR,
    end_year=CORPUS_END_YEAR,
)
corpus = limit_debug_abstracts(corpus, COMMON)
print('Accepted corpus rows:', len(corpus))
if corpus.empty:
    raise ValueError('No accepted abstracts remain after category-stage filtering.')
display(corpus.groupby('pathogen').size().rename('abstracts').reset_index())

Accepted corpus rows: 10


,pathogen,abstracts
0,Bacillus subtilis,5
1,Coxiella burnetii,5


## Classify animal-infection and zoonosis evidence

In [4]:
if not OPENAI_API_KEY:
    raise RuntimeError('Set OPENAI_API_KEY before running the category screen.')
llm = OpenAIChatCompleter(OPENAI_MODEL)
screening_run = screen_pubmed_corpus(
    corpus,
    PATHOGENS,
    llm,
    OUTPUT_DIR,
    model=OPENAI_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    retries=LLM_RETRIES,
    max_llm_calls=LLM_MAX_CALLS,
    save_every=LLM_SAVE_EVERY,
    resume=RESUME,
    retry_failed=RETRY_FAILED_LLM_ROWS,
)
print('Category decisions:', len(screening_run.screening))
print('Classification failures:', len(screening_run.failures))

Screened 1/10: Bacillus subtilis / PMID 23792275
Screened 2/10: Bacillus subtilis / PMID 25208299
Screened 3/10: Bacillus subtilis / PMID 26909865
Screened 4/10: Bacillus subtilis / PMID 27227299
Screened 5/10: Bacillus subtilis / PMID 28421279
Screened 6/10: Coxiella burnetii / PMID 12762362
Screened 7/10: Coxiella burnetii / PMID 17147957
Screened 8/10: Coxiella burnetii / PMID 18755387
Screened 9/10: Coxiella burnetii / PMID 26730641
Screened 10/10: Coxiella burnetii / PMID 27856520
Category decisions: 10
Classification failures: 0


## Derive categories and associate them with pathogens

In [5]:
screened_with_categories, zoonosis_timeline, category_counts = derive_category_counts(
    screening_run.screening,
    OUTPUT_DIR,
)
pathogen_category_table = pd.read_parquet(OUTPUT_DIR / 'pathogen_category_table.parquet')
print('First confirmed zoonosis by pathogen:')
display(zoonosis_timeline)
print('Pathogen category table:')
display(pathogen_category_table)
print('Yearly and total category counts:')
display(category_counts.sort_values(['pathogen', 'publication_year', 'category'], na_position='first'))

First confirmed zoonosis by pathogen:


,pathogen,first_confirmed_zoonosis_year
0,Coxiella burnetii,2003


Pathogen category table:


,pathogen,category_1_count,category_2_count,category_3_count,total_counted,first_confirmed_zoonosis_year
0,Bacillus subtilis,1,0,0,1,NaN
1,Coxiella burnetii,0,0,4,4,2003.0


Yearly and total category counts:


,pathogen,publication_year,category,count
0,Bacillus subtilis,<NA>,1,1
2,Bacillus subtilis,2013,1,1
1,Coxiella burnetii,<NA>,3,4
3,Coxiella burnetii,2003,3,1
4,Coxiella burnetii,2008,3,1
5,Coxiella burnetii,2016,3,1
6,Coxiella burnetii,2017,3,1


Category 1 means no confirmed zoonosis evidence was found in the searched
corpus for that pathogen; it is not proof that the pathogen has never
undergone zoonosis. Review-required and failed rows are excluded from final
counts. The category table is a corpus-level summary, not a biological truth
label.

In [6]:
print('Output directory:', OUTPUT_DIR)
print('Files:', sorted(path.name for path in OUTPUT_DIR.glob('*') if path.is_file()))
print('Review-required rows:', int(screened_with_categories['review_required'].fillna(True).sum()))
display(screened_with_categories[[
    'pathogen', 'pmid', 'llm_category', 'final_category',
    'target_pathogen_supported', 'animal_infection_supported',
    'zoonosis_supported', 'confidence', 'review_required', 'rationale'
]].head(20))

Output directory: /Users/f.costa/Code/RecursiveFraming/outputs/pubmed_screening/debug
Files: ['category_counts.parquet', 'corpus_articles.parquet', 'corpus_articles.parquet.lock', 'llm_refinement.parquet', 'llm_refinement.parquet.lock', 'llm_screening.parquet', 'llm_screening.parquet.lock', 'llm_screening_with_categories.parquet', 'pathogen_category_table.parquet', 'refined_corpus_articles.parquet', 'run_manifest.json', 'search_pmids.parquet', 'search_pmids.parquet.lock', 'search_runs.parquet', 'search_runs.parquet.lock', 'zoonosis_timeline.parquet']
Review-required rows: 4


,pathogen,pmid,llm_category,final_category,target_pathogen_supported,animal_infection_supported,zoonosis_supported,confidence,review_required,rationale
0,Bacillus subtilis,23792275,1,1,True,True,False,0.85,False,The abstract provides clear evidence of Bacill...
1,Bacillus subtilis,25208299,2,<NA>,True,False,False,0.70,True,The abstract discusses research on Bacillus su...
2,Bacillus subtilis,26909865,1,<NA>,True,False,False,0.85,False,The abstract provides clear evidence of antiba...
3,Bacillus subtilis,27227299,2,<NA>,True,False,False,0.70,True,The abstract discusses Bacillus subtilis in th...
4,Bacillus subtilis,28421279,2,<NA>,True,False,False,0.70,True,The abstract discusses Bacillus subtilis in th...
5,Coxiella burnetii,12762362,3,3,True,True,True,1.00,False,The abstract explicitly states that Q fever is...
6,Coxiella burnetii,17147957,2,<NA>,True,True,True,0.70,True,The abstract discusses the host range of Coxie...
7,Coxiella burnetii,18755387,3,3,True,True,True,1.00,False,The abstract explicitly states that Q fever is...
8,Coxiella burnetii,26730641,3,3,True,True,True,0.90,False,The abstract explicitly states that Q fever is...
9,Coxiella burnetii,27856520,3,3,True,True,True,0.90,False,The abstract explicitly states that Coxiella b...
